In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p ~/.kaggle
!cp "/content/drive/MyDrive/Colab Notebooks/kaggle.json" ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Mounted at /content/drive


In [ ]:
# 데이터 구조 변경 함수
import os
import shutil
import random

# 자동 폴더 생성 함수
def make_dirs(base):
    for split in ["train", "val"]:
        for cls in ["real", "ai"]:
            os.makedirs(os.path.join(base, split, cls), exist_ok=True)

# 이미지 분할 + 이동 함수
def split_and_move(src, dst, label, split_ratio=0.8):

    images = []

    # 모든 이미지 수집
    for root, _, files in os.walk(src):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                images.append(os.path.join(root, f))
    random.seed(42)
    random.shuffle(images)

    split_idx = int(len(images) * split_ratio)
    train_imgs = images[:split_idx]
    val_imgs = images[split_idx:]

    def copy_imgs(img_list, split_type):
        for img_path in img_list:
            filename = os.path.basename(img_path)
            dst_path = os.path.join(dst, split_type, label, filename)
            shutil.copy(img_path, dst_path)

    copy_imgs(train_imgs, "train")
    copy_imgs(val_imgs, "val")

    print(f"{label}: train={len(train_imgs)}, val={len(val_imgs)}")

# Phase별 구조 자동 변환 함수
def convert_dataset(base_dir, ai_paths, real_paths, out_dir):

    print("\n" + "="*60)
    print("Processing:", base_dir)
    print("="*60)

    make_dirs(out_dir)

    # AI 데이터
    for p in ai_paths:
        split_and_move(p, out_dir, "ai")

    # REAL 데이터
    for p in real_paths:
        split_and_move(p, out_dir, "real")


# Phase 1 - CIFAKE Dataset

In [ ]:
# Phase 1
# CIFAKE Dataset
# 저해상도 기반 기초 특징 학습

!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images
!unzip -o -q cifake-real-and-ai-generated-synthetic-images.zip -d 1_CIFAKE
!rm cifake-real-and-ai-generated-synthetic-images.zip

Dataset URL: https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images
License(s): other
100% 105M/105M [00:00<00:00, 148MB/s]



In [ ]:
# 데이터셋 구조 확인
import os

base_dir = "/content/1_CIFAKE"

for root, dirs, files in os.walk(base_dir):
    print(root)
    print("폴더:", dirs)
    print("이미지 개수:", len(files))
    print("-" * 50)

/content/1_CIFAKE
폴더: ['train', 'test']
이미지 개수: 0
--------------------------------------------------
/content/1_CIFAKE/train
폴더: ['REAL', 'FAKE']
이미지 개수: 0
--------------------------------------------------
/content/1_CIFAKE/train/REAL
폴더: []
이미지 개수: 50000
--------------------------------------------------
/content/1_CIFAKE/train/FAKE
폴더: []
이미지 개수: 50000
--------------------------------------------------
/content/1_CIFAKE/test
폴더: ['REAL', 'FAKE']
이미지 개수: 0
--------------------------------------------------
/content/1_CIFAKE/test/REAL
폴더: []
이미지 개수: 10000
--------------------------------------------------
/content/1_CIFAKE/test/FAKE
폴더: []
이미지 개수: 10000
--------------------------------------------------


In [ ]:
# 데이터셋 정리

convert_dataset(
    "/content/1_CIFAKE",
    ai_paths=[
        "/content/1_CIFAKE/train/FAKE",
        "/content/1_CIFAKE/test/FAKE"
    ],
    real_paths=[
        "/content/1_CIFAKE/train/REAL",
        "/content/1_CIFAKE/test/REAL"
    ],
    out_dir="/content/clean/Phase1_CIFAKE"
)


Processing: /content/1_CIFAKE
ai: train=40000, val=10000
ai: train=8000, val=2000
real: train=40000, val=10000
real: train=8000, val=2000


In [ ]:
# 재구성한 데이터 구조 확인

!ls /content/clean/Phase1_CIFAKE/train/


# 재구성한 데이터 개수 확인 (중복 제거)

import os

base_dir = "/content/clean/Phase1_CIFAKE"

for split in ["train", "val"]:
    print("\n" + "="*50)
    print(f"📁 {split.upper()}")

    split_path = os.path.join(base_dir, split)

    total = 0

    for cls in ["real", "ai"]:
        class_path = os.path.join(split_path, cls)
        count = len(os.listdir(class_path))
        total += count

        print(f"{cls}: {count}")

    print(f"TOTAL: {total}")

ai  real

📁 TRAIN
real: 41637
ai: 48000
TOTAL: 89637

📁 VAL
real: 11618
ai: 12000
TOTAL: 23618


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import os

# 데이터 경로
train_dir = "/content/clean/Phase1_CIFAKE/train"

# 이미지 사이즈 고정
IMG_SIZE = 128

# 데이터 로딩
def load_dataset(path, img_size, batch_size=32):
    train_ds = tf.keras.preprocessing.image_dataset_from_directory(
        path,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=(img_size, img_size),
        batch_size=batch_size,
        label_mode="int"
    )

    val_ds = tf.keras.preprocessing.image_dataset_from_directory(
        path,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=(img_size, img_size),
        batch_size=batch_size,
        label_mode="int"
    )

    return train_ds, val_ds

In [ ]:
# CNN 모델
def build_model(img_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Rescaling(1./255, input_shape=(img_size, img_size, 3)),

        tf.keras.layers.Conv2D(32, 3, activation='relu'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(64, 3, activation='relu'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(128, 3, activation='relu'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(2, activation='softmax')
    ])
    return model

In [ ]:
# 학습 실행

train_ds, val_ds = load_dataset(train_dir, IMG_SIZE, batch_size=32)

model = build_model(IMG_SIZE)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop]
)

Found 89637 files belonging to 2 classes.
Using 71710 files for training.
Found 89637 files belonging to 2 classes.
Using 17927 files for validation.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
2241/2241 ━━━━━━━━━━━━━━━━━━━━ 56s 22ms/step - accuracy: 0.8211 - loss: 0.3971 - val_accuracy: 0.8838 - val_loss: 0.2843
Epoch 2/20
2241/2241 ━━━━━━━━━━━━━━━━━━━━ 45s 20ms/step - accuracy: 0.9007 - loss: 0.2517 - val_accuracy: 0.9202 - val_loss: 0.2079
Epoch 3/20
2241/2241 ━━━━━━━━━━━━━━━━━━━━ 48s 22ms/step - accuracy: 0.9195 - loss: 0.2074 - val_accuracy: 0.8810 - val_loss: 0.3001
Epoch 4/20
2241/2241 ━━━━━━━━━━━━━━━━━━━━ 82s 22ms/step - accuracy: 0.9307 - loss: 0.1808 - val_accuracy: 0.9340 - val_loss: 0.1785
Epoch 5/20
2241/2241 ━━━━━━━━━━━━━━━━━━━━ 45s 20ms/step - accuracy: 0.9391 - loss: 0.1586 - val_accuracy: 0.9279 - val_loss: 0.1948
Epoch 6/20
2241/2241 ━━━━━━━━━━━━━━━━━━━━ 46s 21ms/step - accuracy: 0.9463 - loss: 0.1418 - val_accuracy: 0.9390 - val_loss: 0.1663
Epoch 7/20
2241/2241 ━━━━━━━━━━━━━━━━━━━━ 48s 21ms/step - accuracy: 0.9502 - loss: 0.1319 - val_accuracy: 0.9308 - val_loss: 0.1967
Epoch 8/20
2241/2241 ━━━━━━━━━━━━━━━━━━━━ 46s 21ms/step - accuracy: 0.9556 -

In [ ]:
# model save

model.save("/content/drive/MyDrive/phase1.keras")

# Phase 2 - AI Generated Images vs Real Images Dataset

In [ ]:
# Phase 2
# AI Generated Images vs Real Images Dataset
# 일반 이미지 기반 초기 일반화

!kaggle datasets download -d cashbowman/ai-generated-images-vs-real-images
!unzip -o -q ai-generated-images-vs-real-images.zip -d 2_AI_Generated_vs_Real
!rm ai-generated-images-vs-real-images.zip

Dataset URL: https://www.kaggle.com/datasets/cashbowman/ai-generated-images-vs-real-images
License(s): Community Data License Agreement - Sharing - Version 1.0
100% 476M/476M [00:08<00:00, 60.9MB/s]



In [ ]:
# 데이터셋 구조 확인
base_dir = "/content/2_AI_Generated_vs_Real"

for root, dirs, files in os.walk(base_dir):
    print(root)
    print("폴더:", dirs)
    print("이미지 개수:", len(files))
    print("-" * 50)

/content/2_AI_Generated_vs_Real
폴더: ['AiArtData', 'RealArt']
이미지 개수: 0
--------------------------------------------------
/content/2_AI_Generated_vs_Real/AiArtData
폴더: ['AiArtData']
이미지 개수: 0
--------------------------------------------------
/content/2_AI_Generated_vs_Real/AiArtData/AiArtData
폴더: []
이미지 개수: 539
--------------------------------------------------
/content/2_AI_Generated_vs_Real/RealArt
폴더: ['RealArt']
이미지 개수: 0
--------------------------------------------------
/content/2_AI_Generated_vs_Real/RealArt/RealArt
폴더: []
이미지 개수: 436
--------------------------------------------------


In [ ]:
# 데이터셋 정리

convert_dataset(
    "/content/2_AI_Generated_vs_Real",
    ai_paths=["/content/2_AI_Generated_vs_Real/AiArtData/AiArtData"],
    real_paths=["/content/2_AI_Generated_vs_Real/RealArt/RealArt"],
    out_dir="/content/clean/Phase2_AI_REAL"
)


Processing: /content/2_AI_Generated_vs_Real
ai: train=431, val=108
real: train=347, val=87


In [ ]:
# 재구성한 데이터 구조 확인

!ls /content/clean/Phase2_AI_REAL/


# 재구성한 데이터 개수 확인 (중복 제거)

import os

base_dir = "/content/clean/Phase2_AI_REAL"

for split in ["train", "val"]:
    print("\n" + "="*50)
    print(f"📁 {split.upper()}")

    split_path = os.path.join(base_dir, split)

    total = 0

    for cls in ["real", "ai"]:
        class_path = os.path.join(split_path, cls)
        count = len(os.listdir(class_path))
        total += count

        print(f"{cls}: {count}")

    print(f"TOTAL: {total}")

train  val

📁 TRAIN
real: 347
ai: 431
TOTAL: 778

📁 VAL
real: 87
ai: 108
TOTAL: 195


In [ ]:
# 주소 변경
train_dir = "/content/clean/Phase2_AI_REAL/train"

train_ds, val_ds = load_dataset(train_dir, IMG_SIZE, batch_size=32)

Found 776 files belonging to 2 classes.
Using 621 files for training.
Found 776 files belonging to 2 classes.
Using 155 files for validation.


In [ ]:
# 모델 불러오기
model = tf.keras.models.load_model(
    "/content/drive/MyDrive/phase1.keras"
)

# 모델 학습
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop]
)

# 모델 저장
model.save("/content/drive/MyDrive/phase2_a.keras")

Epoch 1/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 21s 756ms/step - accuracy: 0.5411 - loss: 24.7983 - val_accuracy: 0.6065 - val_loss: 16.1419
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 11s 572ms/step - accuracy: 0.5588 - loss: 11.6650 - val_accuracy: 0.6516 - val_loss: 7.7603
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 13s 654ms/step - accuracy: 0.6039 - loss: 5.7140 - val_accuracy: 0.6129 - val_loss: 4.0154
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 21s 681ms/step - accuracy: 0.6055 - loss: 2.9960 - val_accuracy: 0.5806 - val_loss: 2.8681
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 14s 716ms/step - accuracy: 0.5749 - loss: 2.4837 - val_accuracy: 0.4968 - val_loss: 2.4336
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 19s 628ms/step - accuracy: 0.5668 - loss: 1.9687 - val_accuracy: 0.4839 - val_loss: 2.2019
Epoch 7/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 21s 638ms/step - accuracy: 0.5717 - loss: 1.7719 - val_accuracy: 0.4839 - val_loss: 1.9824
Epoch 8/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 12s 605ms/step - accuracy: 0.5684 - loss: 1.6428 - val_a

In [ ]:
# 모델 불러오기
model = tf.keras.models.load_model(
    "/content/drive/MyDrive/phase1.keras"
)

# 데이터 증강
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
])

def aug(x, y):
    x = tf.cast(x, tf.float32) / 255.0
    x = data_aug(x, training=True)
    return x, y

def norm(x, y):
    x = tf.cast(x, tf.float32) / 255.0
    return x, y

# 데이터셋 적용
train_ds_aug = train_ds.map(
    aug,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_ds_aug = val_ds.map(
    norm,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_ds_aug = train_ds_aug.cache().prefetch(tf.data.AUTOTUNE)
val_ds_aug = val_ds_aug.cache().prefetch(tf.data.AUTOTUNE)

# 모델 학습
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history_phase2_b = model.fit(
    train_ds_aug,
    validation_data=val_ds_aug,
    epochs=20,
    callbacks=[early_stop]
)

# 모델 저장
model.save("/content/drive/MyDrive/phase2_b.keras")

Epoch 1/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 21s 722ms/step - accuracy: 0.4589 - loss: 19.8622 - val_accuracy: 0.4000 - val_loss: 19.2036
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.4589 - loss: 15.1930 - val_accuracy: 0.4000 - val_loss: 13.6190
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.4589 - loss: 10.7045 - val_accuracy: 0.4000 - val_loss: 9.2762
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.4589 - loss: 7.0088 - val_accuracy: 0.4000 - val_loss: 5.5203
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.4605 - loss: 3.7545 - val_accuracy: 0.4000 - val_loss: 3.1005
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.4605 - loss: 2.4155 - val_accuracy: 0.4000 - val_loss: 2.1007
Epoch 7/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.4605 - loss: 1.6445 - val_accuracy: 0.4000 - val_loss: 1.3411
Epoch 8/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.4638 - loss: 1.0873 - val_accuracy: 0.4

In [ ]:
# 모델 선택하기

import os

os.rename(
    "/content/drive/MyDrive/phase2_b.keras",
    "/content/drive/MyDrive/phase2.keras"
)

# Phase 3 - AI vs Real Images Dataset

In [ ]:
# Phase 3
# AI vs Real Images Dataset
# SDXL 기반 고품질 이미지 학습

!kaggle datasets download -d rhythmghai/ai-vs-real-images-dataset
!unzip -o -q ai-vs-real-images-dataset.zip -d 3_AI_vs_Real
!rm ai-vs-real-images-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/rhythmghai/ai-vs-real-images-dataset
License(s): CC-BY-NC-SA-4.0
100% 237M/237M [00:02<00:00, 112MB/s]



In [ ]:
# 데이터셋 구조 확인
base_dir = "/content/3_AI_vs_Real"

for root, dirs, files in os.walk(base_dir):
    print(root)
    print("폴더:", dirs)
    print("이미지 개수:", len(files))
    print("-" * 50)

/content/3_AI_vs_Real
폴더: ['Ai_generated_dataset', 'real_dataset']
이미지 개수: 0
--------------------------------------------------
/content/3_AI_vs_Real/Ai_generated_dataset
폴더: ['city', 'animals', 'nature', 'people', 'food']
이미지 개수: 0
--------------------------------------------------
/content/3_AI_vs_Real/Ai_generated_dataset/city
폴더: []
이미지 개수: 50
--------------------------------------------------
/content/3_AI_vs_Real/Ai_generated_dataset/animals
폴더: []
이미지 개수: 50
--------------------------------------------------
/content/3_AI_vs_Real/Ai_generated_dataset/nature
폴더: []
이미지 개수: 50
--------------------------------------------------
/content/3_AI_vs_Real/Ai_generated_dataset/people
폴더: []
이미지 개수: 50
--------------------------------------------------
/content/3_AI_vs_Real/Ai_generated_dataset/food
폴더: []
이미지 개수: 50
--------------------------------------------------
/content/3_AI_vs_Real/real_dataset
폴더: ['city', 'animals', 'nature', 'people', 'food']
이미지 개수: 0
---------------------------

In [ ]:
# 데이터셋 정리

convert_dataset(
    base_dir="/content/3_AI_vs_Real",
    ai_paths=["/content/3_AI_vs_Real/Ai_generated_dataset"],
    real_paths=["/content/3_AI_vs_Real/real_dataset"],
    out_dir="/content/clean/Phase3_AI_REAL"
)


Processing: /content/3_AI_vs_Real
ai: train=200, val=50
real: train=596, val=149


In [ ]:
# 재구성한 데이터 구조 확인

!ls /content/clean/Phase3_AI_REAL/


# 재구성한 데이터 개수 확인 (중복 제거)

import os

base_dir = "/content/clean/Phase3_AI_REAL"

for split in ["train", "val"]:
    print("\n" + "="*50)
    print(f"📁 {split.upper()}")

    split_path = os.path.join(base_dir, split)

    total = 0

    for cls in ["real", "ai"]:
        class_path = os.path.join(split_path, cls)
        count = len(os.listdir(class_path))
        total += count

        print(f"{cls}: {count}")

    print(f"TOTAL: {total}")

train  val

📁 TRAIN
real: 596
ai: 50
TOTAL: 646

📁 VAL
real: 149
ai: 31
TOTAL: 180


In [ ]:
# 주소 변경
train_dir = "/content/clean/Phase3_AI_REAL/train"

train_ds, val_ds = load_dataset(train_dir, IMG_SIZE, batch_size=32)

Found 646 files belonging to 2 classes.
Using 517 files for training.
Found 646 files belonging to 2 classes.
Using 129 files for validation.


In [ ]:
# 모델 불러오기
model = tf.keras.models.load_model(
    "/content/drive/MyDrive/phase1.keras"
)

# 모델 학습
model.compile(
    optimizer=tf.keras.optimizers.RMSprop(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history_phase3 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop]
)

# 실험 결과 저장
model.save("/content/drive/MyDrive/phase3_a.keras")

Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 14s 535ms/step - accuracy: 0.1335 - loss: 35.2527 - val_accuracy: 0.2016 - val_loss: 25.5600
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 528ms/step - accuracy: 0.2263 - loss: 16.4774 - val_accuracy: 0.2636 - val_loss: 13.7140
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 8s 472ms/step - accuracy: 0.3907 - loss: 8.2351 - val_accuracy: 0.4264 - val_loss: 6.8738
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 398ms/step - accuracy: 0.5648 - loss: 4.0131 - val_accuracy: 0.6047 - val_loss: 3.4784
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 10s 607ms/step - accuracy: 0.7041 - loss: 1.8594 - val_accuracy: 0.7364 - val_loss: 1.8271
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 7s 428ms/step - accuracy: 0.7911 - loss: 1.2448 - val_accuracy: 0.8062 - val_loss: 1.1169
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 10s 393ms/step - accuracy: 0.8569 - loss: 0.8283 - val_accuracy: 0.8450 - val_loss: 0.7370
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 10s 588ms/step - accuracy: 0.8839 - loss: 0.7727 - val_accu

In [ ]:
# 모델 불러오기
model = tf.keras.models.load_model(
    "/content/drive/MyDrive/phase1.keras"
)

# 추가
# 데이터 증강
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
])
def aug(x, y):
    x = x / 255.0
    x = data_aug(x, training=True)
    return x, y

def norm(x, y):
    return x / 255.0, y

train_ds_aug = train_ds.map(aug)
val_ds_aug = val_ds.map(norm)


# 모델 학습
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history_phase3 = model.fit(
    train_ds_aug,
    validation_data=val_ds_aug,
    epochs=20,
    callbacks=[early_stop]
)

# 실험 결과 저장
model.save("/content/drive/MyDrive/phase3_b.keras")



Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 17s 713ms/step - accuracy: 0.9207 - loss: 3.0486 - val_accuracy: 0.9302 - val_loss: 2.4651
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 669ms/step - accuracy: 0.9207 - loss: 2.6609 - val_accuracy: 0.9302 - val_loss: 1.9167
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 10s 561ms/step - accuracy: 0.9207 - loss: 2.1753 - val_accuracy: 0.9302 - val_loss: 1.4608
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 10s 544ms/step - accuracy: 0.9207 - loss: 1.4735 - val_accuracy: 0.9302 - val_loss: 1.1084
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 12s 684ms/step - accuracy: 0.9207 - loss: 1.0757 - val_accuracy: 0.9302 - val_loss: 0.7612
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 12s 714ms/step - accuracy: 0.9207 - loss: 0.6861 - val_accuracy: 0.9302 - val_loss: 0.4633
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 542ms/step - accuracy: 0.9207 - loss: 0.4776 - val_accuracy: 0.9302 - val_loss: 0.3659
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 461ms/step - accuracy: 0.9207 - loss: 0.3649 - val_accura

In [ ]:
# 모델 선택하기

import os

os.rename(
    "/content/drive/MyDrive/phase3_a.keras",
    "/content/drive/MyDrive/phase3.keras"
)

# Phase 4 - AI vs Human Generated Dataset

In [ ]:
# Phase 4
# AI vs Human Generated Dataset

!kaggle datasets download -d alessandrasala79/ai-vs-human-generated-dataset
!unzip -o -q ai-vs-human-generated-dataset.zip -d 4_AI_vs_Human
!rm ai-vs-human-generated-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/alessandrasala79/ai-vs-human-generated-dataset
License(s): apache-2.0
100% 9.76G/9.76G [02:21<00:00, 74.1MB/s]



In [ ]:
# 데이터셋 구조 확인
base_dir = "/content/4_AI_vs_Human"

for root, dirs, files in os.walk(base_dir):
    print(root)
    print("폴더:", dirs)
    print("이미지 개수:", len(files))
    print("-" * 50)

/content/4_AI_vs_Human
폴더: ['test_data_v2', 'train_data']
이미지 개수: 2
--------------------------------------------------
/content/4_AI_vs_Human/test_data_v2
폴더: []
이미지 개수: 5540
--------------------------------------------------
/content/4_AI_vs_Human/train_data
폴더: []
이미지 개수: 79950
--------------------------------------------------


In [ ]:
import os
import shutil
import random
import pandas as pd

# 폴더 생성
def make_dirs(base):
    for split in ["train", "val"]:
        for cls in ["real", "ai"]:
            os.makedirs(os.path.join(base, split, cls), exist_ok=True)

# CSV 기반 split + move
def split_and_move_from_csv(df, base_dir, out_dir, label_name, split_ratio=0.8):

    rows = df.to_dict("records")

    random.seed(42)
    random.shuffle(rows)

    split_idx = int(len(rows) * split_ratio)
    train_rows = rows[:split_idx]
    val_rows = rows[split_idx:]

    def copy_rows(row_list, split_type):
        for row in row_list:

            rel_path = row["file_name"]   # CSV 컬럼 기준
            src_path = os.path.join(base_dir, rel_path)

            filename = os.path.basename(rel_path)
            dst_path = os.path.join(out_dir, split_type, label_name, filename)

            if os.path.exists(src_path):
                shutil.copy2(src_path, dst_path)

    copy_rows(train_rows, "train")
    copy_rows(val_rows, "val")

    print(f"{label_name}: train={len(train_rows)}, val={len(val_rows)}")

# 전체 변환 함수 (CSV 버전)
def convert_dataset(base_dir, csv_path, out_dir):

    print("\n" + "="*60)
    print("Processing:", base_dir)
    print("="*60)

    make_dirs(out_dir)

    df = pd.read_csv(csv_path)

    # label 분리
    real_df = df[df["label"] == 0]
    ai_df   = df[df["label"] == 1]

    print("REAL:", len(real_df))
    print("AI:", len(ai_df))

    # 각각 처리
    split_and_move_from_csv(real_df, base_dir, out_dir, "real")
    split_and_move_from_csv(ai_df, base_dir, out_dir, "ai")

In [ ]:
# 데이터셋 정리

convert_dataset(
    base_dir="/content/4_AI_vs_Human",
    csv_path="/content/4_AI_vs_Human/train.csv",
    out_dir="/content/clean/Phase4_AI_REAL"
)


Processing: /content/4_AI_vs_Human
REAL: 39975
AI: 39975
real: train=31980, val=7995
ai: train=31980, val=7995


In [ ]:
# 재구성한 데이터 구조 확인

!ls /content/clean/Phase4_AI_REAL/


# 재구성한 데이터 개수 확인 (중복 제거)

import os

base_dir = "/content/clean/Phase4_AI_REAL"

for split in ["train", "val"]:
    print("\n" + "="*50)
    print(f"📁 {split.upper()}")

    split_path = os.path.join(base_dir, split)

    total = 0

    for cls in ["real", "ai"]:
        class_path = os.path.join(split_path, cls)
        count = len(os.listdir(class_path))
        total += count

        print(f"{cls}: {count}")

    print(f"TOTAL: {total}")

train  val

📁 TRAIN
real: 31980
ai: 31980
TOTAL: 63960

📁 VAL
real: 7995
ai: 7995
TOTAL: 15990


In [ ]:
# 주소 변경
train_dir = "/content/clean/Phase4_AI_REAL/train"

train_ds, val_ds = load_dataset(train_dir, IMG_SIZE, batch_size=32)


train_ds = train_ds.ignore_errors()

Found 63960 files belonging to 2 classes.
Using 51168 files for training.
Found 63960 files belonging to 2 classes.
Using 12792 files for validation.


In [ ]:
# 모델 불러오기
model = tf.keras.models.load_model(
    "/content/drive/MyDrive/phase3.keras"
)

# 모델 학습

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_phase4_no_aug = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20
)


# 실험 결과 저장
model.save("/content/drive/MyDrive/phase4_a.keras")

Epoch 1/20
   1598/Unknown 110s 67ms/step - accuracy: 0.7477 - loss: 0.7063

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1598/1598 ━━━━━━━━━━━━━━━━━━━━ 141s 86ms/step - accuracy: 0.8140 - loss: 0.4717 - val_accuracy: 0.8926 - val_loss: 0.2855
Epoch 2/20
1598/1598 ━━━━━━━━━━━━━━━━━━━━ 128s 80ms/step - accuracy: 0.9005 - loss: 0.2697 - val_accuracy: 0.9064 - val_loss: 0.2431
Epoch 3/20
1598/1598 ━━━━━━━━━━━━━━━━━━━━ 113s 71ms/step - accuracy: 0.9146 - loss: 0.2332 - val_accuracy: 0.9208 - val_loss: 0.2085
Epoch 4/20
1598/1598 ━━━━━━━━━━━━━━━━━━━━ 114s 71ms/step - accuracy: 0.9233 - loss: 0.2067 - val_accuracy: 0.9192 - val_loss: 0.2184
Epoch 5/20
1598/1598 ━━━━━━━━━━━━━━━━━━━━ 113s 70ms/step - accuracy: 0.9299 - loss: 0.1898 - val_accuracy: 0.9334 - val_loss: 0.1828
Epoch 6/20
1598/1598 ━━━━━━━━━━━━━━━━━━━━ 136s 85ms/step - accuracy: 0.9360 - loss: 0.1749 - val_accuracy: 0.9370 - val_loss: 0.1719
Epoch 7/20
1598/1598 ━━━━━━━━━━━━━━━━━━━━ 111s 70ms/step - accuracy: 0.9397 - loss: 0.1649 - val_accuracy: 0.9403 - val_loss: 0.1623
Epoch 8/20
1598/1598 ━━━━━━━━━━━━━━━━━━━━ 110s 69ms/step - accuracy: 0.9442 - lo

In [ ]:
# 모델 선택하기

import os

os.rename(
    "/content/drive/MyDrive/phase4_a.keras",
    "/content/drive/MyDrive/phase4.keras"
)

# Phase 5 - Real vs AI Generated Faces Dataset

In [ ]:
# Phase 5
# Real vs AI Generated Faces Dataset
# 고난도 얼굴 생성 탐지

!kaggle datasets download -d philosopher0808/real-vs-ai-generated-faces-dataset
!unzip -o -q real-vs-ai-generated-faces-dataset.zip -d 5_Real_vs_AI_Faces
!rm real-vs-ai-generated-faces-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/philosopher0808/real-vs-ai-generated-faces-dataset
License(s): MIT
100% 7.19G/7.19G [00:57<00:00, 133MB/s]



In [ ]:
# 데이터셋 구조 확인
import os

base_dir = "/content/5_Real_vs_AI_Faces"

for root, dirs, files in os.walk(base_dir):
    print(root)
    print("폴더:", dirs)
    print("이미지 개수:", len(files))
    print("-" * 50)

/content/5_Real_vs_AI_Faces
폴더: ['data_source', 'dataset']
이미지 개수: 0
--------------------------------------------------
/content/5_Real_vs_AI_Faces/data_source
폴더: ['data_source']
이미지 개수: 0
--------------------------------------------------
/content/5_Real_vs_AI_Faces/data_source/data_source
폴더: ['ffhq', 'fake']
이미지 개수: 0
--------------------------------------------------
/content/5_Real_vs_AI_Faces/data_source/data_source/ffhq
폴더: []
이미지 개수: 70000
--------------------------------------------------
/content/5_Real_vs_AI_Faces/data_source/data_source/fake
폴더: ['stable_diffusion', 'sfhq', 'faceswap', 'thispersondoesnotexist']
이미지 개수: 0
--------------------------------------------------
/content/5_Real_vs_AI_Faces/data_source/data_source/fake/stable_diffusion
폴더: []
이미지 개수: 1031
--------------------------------------------------
/content/5_Real_vs_AI_Faces/data_source/data_source/fake/sfhq
폴더: ['pt4', 'pt1', 'pt3', 'pt2']
이미지 개수: 0
--------------------------------------------------
/conte

In [ ]:
# 데이터셋 정리

import os
import shutil
import random

base_dir = "/content/5_Real_vs_AI_Faces/data_source/data_source"
out_dir = "/content/clean/Phase5_AI_REAL"

# 폴더 생성
for split in ["train", "val"]:
    for cls in ["real", "ai"]:
        os.makedirs(os.path.join(out_dir, split, cls), exist_ok=True)

# 경로 정의
real_path = os.path.join(base_dir, "ffhq")

ai_paths = [
    "fake/stable_diffusion",
    "fake/sfhq/pt1",
    "fake/sfhq/pt2",
    "fake/sfhq/pt3",
    "fake/sfhq/pt4",
    "fake/faceswap",
    "fake/thispersondoesnotexist"
]

# 이미지 수집 함수 (inline)
def collect_images(folder):
    imgs = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                imgs.append(os.path.join(root, f))
    return imgs

# REAL
real_imgs = collect_images(real_path)

# AI
ai_imgs = []
for p in ai_paths:
    full_path = os.path.join(base_dir, p)
    ai_imgs.extend(collect_images(full_path))

# shuffle + split
random.seed(42)

random.shuffle(real_imgs)
random.shuffle(ai_imgs)

real_split = int(len(real_imgs) * 0.8)
ai_split = int(len(ai_imgs) * 0.8)

real_train, real_val = real_imgs[:real_split], real_imgs[real_split:]
ai_train, ai_val = ai_imgs[:ai_split], ai_imgs[ai_split:]

# 복사 함수
def copy_files(file_list, split, label):
    for path in file_list:
        filename = os.path.basename(path)
        dst = os.path.join(out_dir, split, label, filename)
        shutil.copy2(path, dst)

# 실행
copy_files(real_train, "train", "real")
copy_files(real_val, "val", "real")

copy_files(ai_train, "train", "ai")
copy_files(ai_val, "val", "ai")

# 결과 출력
print("REAL:", len(real_imgs), "AI:", len(ai_imgs))
print("DONE")

REAL: 70000 AI: 50959
DONE


In [ ]:
# 재구성한 데이터 구조 확인

!ls /content/clean/Phase5_AI_REAL/


# 재구성한 데이터 개수 확인 (중복 제거)

import os

base_dir = "/content/clean/Phase5_AI_REAL"

for split in ["train", "val"]:
    print("\n" + "="*50)
    print(f"📁 {split.upper()}")

    split_path = os.path.join(base_dir, split)

    total = 0

    for cls in ["real", "ai"]:
        class_path = os.path.join(split_path, cls)
        count = len(os.listdir(class_path))
        total += count

        print(f"{cls}: {count}")

    print(f"TOTAL: {total}")

train  val

📁 TRAIN
real: 56000
ai: 40767
TOTAL: 96767

📁 VAL
real: 14000
ai: 10192
TOTAL: 24192


In [ ]:
# 주소 변경
train_dir = "/content/clean/Phase5_AI_REAL/train"

train_ds, val_ds = load_dataset(train_dir, IMG_SIZE, batch_size=32)

Found 96767 files belonging to 2 classes.
Using 77414 files for training.
Found 96767 files belonging to 2 classes.
Using 19353 files for validation.


In [ ]:
# 모델 불러오기
model = tf.keras.models.load_model(
    "/content/drive/MyDrive/phase4.keras"
)

# 모델 학습
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history_phase5 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop]
)

# 실험 결과 저장
model.save("/content/drive/MyDrive/phase5_a.keras")

Epoch 1/20
2420/2420 ━━━━━━━━━━━━━━━━━━━━ 169s 67ms/step - accuracy: 0.6416 - loss: 0.6661 - val_accuracy: 0.7334 - val_loss: 0.5560
Epoch 2/20
2420/2420 ━━━━━━━━━━━━━━━━━━━━ 175s 58ms/step - accuracy: 0.7279 - loss: 0.5497 - val_accuracy: 0.7587 - val_loss: 0.5045
Epoch 3/20
2420/2420 ━━━━━━━━━━━━━━━━━━━━ 118s 49ms/step - accuracy: 0.7596 - loss: 0.5028 - val_accuracy: 0.7826 - val_loss: 0.4694
Epoch 4/20
2420/2420 ━━━━━━━━━━━━━━━━━━━━ 117s 49ms/step - accuracy: 0.7831 - loss: 0.4673 - val_accuracy: 0.8121 - val_loss: 0.4233
Epoch 5/20
2420/2420 ━━━━━━━━━━━━━━━━━━━━ 163s 57ms/step - accuracy: 0.8018 - loss: 0.4358 - val_accuracy: 0.8188 - val_loss: 0.4118
Epoch 6/20
2420/2420 ━━━━━━━━━━━━━━━━━━━━ 119s 49ms/step - accuracy: 0.8187 - loss: 0.4080 - val_accuracy: 0.8348 - val_loss: 0.3846
Epoch 7/20
2420/2420 ━━━━━━━━━━━━━━━━━━━━ 139s 57ms/step - accuracy: 0.8318 - loss: 0.3836 - val_accuracy: 0.8506 - val_loss: 0.3543
Epoch 8/20
2420/2420 ━━━━━━━━━━━━━━━━━━━━ 125s 50ms/step - accuracy: 

In [ ]:
# 모델 선택하기

import os

os.rename(
    "/content/drive/MyDrive/phase5_a.keras",
    "/content/drive/MyDrive/phase5.keras"
)

# 테스트 데이터셋

In [ ]:
# 기존 테스트 데이터셋

import os
import shutil

# 테스트 데이터 저장 폴더 생성
test_base = "/content/test_val"
os.makedirs(f"{test_base}/ai", exist_ok=True)
os.makedirs(f"{test_base}/real", exist_ok=True)

# 각 Phase val 데이터 복사
phases = [
    "/content/clean/Phase1_CIFAKE/val",
    "/content/clean/Phase3_AI_REAL/val",
    "/content/clean/Phase4_AI_REAL/val",
    "/content/clean/Phase5_AI_REAL/val",
]

for phase_val in phases:
    for cls in ["ai", "real"]:
        src = f"{phase_val}/{cls}"
        dst = f"{test_base}/{cls}"
        if os.path.exists(src):
            for f in os.listdir(src):
                shutil.copy(os.path.join(src, f), os.path.join(dst, f))

# 확인
for cls in ["ai", "real"]:
    count = len(os.listdir(f"{test_base}/{cls}"))
    print(f"{cls}: {count}장")

ai: 30218장
real: 33744장


In [ ]:
import kagglehub
import shutil
import os

# Real 이미지 다운로드
real_path = kagglehub.dataset_download("duyminhle/real-image-dataset")
shutil.copytree(real_path, "/content/test_real", dirs_exist_ok=True)
print("Real 이미지 경로:", "/content/test_real")

# AI 이미지 다운로드 (SynthBuster)
fake_path = kagglehub.dataset_download("devpatel484/synthbuster-32")
shutil.copytree(fake_path, "/content/test_fake", dirs_exist_ok=True)
print("AI 이미지 경로:", "/content/test_fake")

# 구조 확인
for label, path in [("REAL", "/content/test_real"), ("FAKE", "/content/test_fake")]:
    print(f"\n--- {label} ---")
    for root, dirs, files in os.walk(path):
        imgs = [f for f in files if f.lower().endswith(('.jpg','.png','.jpeg','.webp'))]
        if imgs:
            print(f"{root}: {len(imgs)}장")

100%|██████████| 893M/893M [00:10<00:00, 92.1MB/s]

Extracting files...


Real 이미지 경로: /content/test_real


100%|██████████| 9.04M/9.04M [00:00<00:00, 78.6MB/s]

Extracting files...


AI 이미지 경로: /content/test_fake

--- REAL ---
/content/test_real/images: 9000장

--- FAKE ---
/content/test_fake/resized_data_Synthbuster/Synthbuster_Dataset/stable-diffusion-1-3: 1000장
/content/test_fake/resized_data_Synthbuster/Synthbuster_Dataset/firefly: 1000장
/content/test_fake/resized_data_Synthbuster/Synthbuster_Dataset/stable-diffusion-2: 1000장
/content/test_fake/resized_data_Synthbuster/Synthbuster_Dataset/dalle3: 1000장
/content/test_fake/resized_data_Synthbuster/Synthbuster_Dataset/dalle2: 1000장
/content/test_fake/resized_data_Synthbuster/Synthbuster_Dataset/glide: 1000장
/content/test_fake/resized_data_Synthbuster/Synthbuster_Dataset/stable-diffusion-1-4: 1000장
/content/test_fake/resized_data_Synthbuster/Synthbuster_Dataset/stable-diffusion-xl: 1000장
/content/test_fake/resized_data_Synthbuster/Synthbuster_Dataset/midjourney-v5: 1000장


In [ ]:
all_labels = [0] * len(fake_path) + [1] * len(real_path)  # ai=0, real=1 정수로

import tensorflow as tf
from pathlib import Path

def load_test_data_v1(real_dir, fake_dir, image_size=(128, 128), batch_size=32):
    real_paths = list(Path(real_dir).glob("*.jpg")) + \
                 list(Path(real_dir).glob("*.png")) + \
                 list(Path(real_dir).glob("*.jpeg"))

    fake_paths = list(Path(fake_dir).glob("**/*.jpg")) + \
                 list(Path(fake_dir).glob("**/*.png")) + \
                 list(Path(fake_dir).glob("**/*.jpeg")) + \
                 list(Path(fake_dir).glob("**/*.webp"))

    print(f"Real: {len(real_paths)}장, Fake: {len(fake_paths)}장")

    all_paths = [str(p) for p in fake_paths + real_paths]
    all_labels = [0] * len(fake_paths) + [1] * len(real_paths)

    def preprocess(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, image_size)
        img = tf.cast(img, tf.float32)  # /255 없음
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((all_paths, all_labels))
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

test_ds_v1 = load_test_data_v1(
    real_dir="/content/test_real/images",
    fake_dir="/content/test_fake/resized_data_Synthbuster/Synthbuster_Dataset",
)

Real: 9000장, Fake: 9000장


In [ ]:
# 누적 학습 모델 평가

from tensorflow.keras.models import load_model

# 누적 학습 모델 로드
sequential_model = load_model("/content/drive/MyDrive/phase5.keras")

# SynthBuster로 테스트
test_loss, test_acc = sequential_model.evaluate(test_ds_v1)

print(f"단계적 학습 모델 test_acc: {test_acc:.4f} / test_loss: {test_loss:.4f}")

# 결과 저장
results_sequential = {
    "dataset": "sequential_phase1_to_5",
    "test_acc": float(test_acc),
    "test_loss": float(test_loss)
}

563/563 ━━━━━━━━━━━━━━━━━━━━ 157s 278ms/step - accuracy: 0.7854 - loss: 1.1971
단계적 학습 모델 test_acc: 0.7854 / test_loss: 1.1971


In [ ]:
import tensorflow as tf
import os
from pathlib import Path

def load_test_data_v2(real_dir, fake_dir, image_size=(128, 128), batch_size=32):
    real_paths = list(Path(real_dir).glob("*.jpg")) + \
                 list(Path(real_dir).glob("*.png")) + \
                 list(Path(real_dir).glob("*.jpeg"))

    fake_paths = list(Path(fake_dir).glob("**/*.jpg")) + \
                 list(Path(fake_dir).glob("**/*.png")) + \
                 list(Path(fake_dir).glob("**/*.jpeg")) + \
                 list(Path(fake_dir).glob("**/*.webp"))

    print(f"Real: {len(real_paths)}장, Fake: {len(fake_paths)}장")

    all_paths = [str(p) for p in fake_paths + real_paths]
    all_labels = [0] * len(fake_paths) + [1] * len(real_paths)  # ai=0, real=1 정수

    def preprocess(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, image_size)
        img = tf.cast(img, tf.float32)  # /255 제거 (모델 안에서 Rescaling이 함)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((all_paths, all_labels))
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return ds

test_ds_v2 = load_test_data_v2(
    real_dir="/content/test_real/images",
    fake_dir="/content/test_fake/resized_data_Synthbuster/Synthbuster_Dataset",
)

Real: 9000장, Fake: 9000장


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model

# val 데이터 로더 (모델 안에 Rescaling 있으니 /255 없음)
def load_test_val(base_dir, image_size=(128, 128), batch_size=32):
    from pathlib import Path

    ai_paths = list(Path(f"{base_dir}/ai").glob("*.jpg")) + \
               list(Path(f"{base_dir}/ai").glob("*.png")) + \
               list(Path(f"{base_dir}/ai").glob("*.jpeg"))

    real_paths = list(Path(f"{base_dir}/real").glob("*.jpg")) + \
                 list(Path(f"{base_dir}/real").glob("*.png")) + \
                 list(Path(f"{base_dir}/real").glob("*.jpeg"))

    print(f"AI: {len(ai_paths)}장, Real: {len(real_paths)}장")

    all_paths = [str(p) for p in ai_paths + real_paths]
    all_labels = [0] * len(ai_paths) + [1] * len(real_paths)  # ai=0, real=1

    def preprocess(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, image_size)
        img = tf.cast(img, tf.float32)  # /255 없음
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((all_paths, all_labels))
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# 누적 학습 모델 로드
sequential_model = load_model("/content/drive/MyDrive/phase5.keras")

test_ds_val = load_test_val("/content/test_val")

# 누적 모델 평가
test_loss, test_acc = sequential_model.evaluate(test_ds_val)
print(f"단계적 학습 모델 test_acc: {test_acc:.4f} / test_loss: {test_loss:.4f}")

# 저장
results_sequential = {
    "dataset": "sequential_phase1_to_5",
    "test_acc": float(test_acc),
    "test_loss": float(test_loss)
}

AI: 30218장, Real: 33744장
1999/1999 ━━━━━━━━━━━━━━━━━━━━ 483s 239ms/step - accuracy: 0.6394 - loss: 4.3325
단계적 학습 모델 test_acc: 0.6394 / test_loss: 4.3325


NameError: name 'SAVE_DIR' is not defined

In [ ]:
import tensorflow as tf
from pathlib import Path

def load_test_val(base_dir, image_size=(128, 128), batch_size=32):
    ai_paths = list(Path(f"{base_dir}/ai").glob("*.jpg")) + \
               list(Path(f"{base_dir}/ai").glob("*.png")) + \
               list(Path(f"{base_dir}/ai").glob("*.jpeg"))

    real_paths = list(Path(f"{base_dir}/real").glob("*.jpg")) + \
                 list(Path(f"{base_dir}/real").glob("*.png")) + \
                 list(Path(f"{base_dir}/real").glob("*.jpeg"))

    print(f"AI: {len(ai_paths)}장, Real: {len(real_paths)}장")

    all_paths = [str(p) for p in ai_paths + real_paths]
    all_labels = [0] * len(ai_paths) + [1] * len(real_paths)

    def preprocess(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, image_size)
        img = tf.cast(img, tf.float32)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((all_paths, all_labels))
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# val 테스트셋 (누적 모델용)
test_ds_v2 = load_test_val("/content/test_val")

AI: 30218장, Real: 33744장


# 개별 데이터셋 모델

In [ ]:
import os
import json
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = 128
EPOCHS = 20
SAVE_DIR = "/content/drive/MyDrive/model_results"
os.makedirs(SAVE_DIR, exist_ok=True)

def build_cnn(img_size=128):
    model = tf.keras.Sequential([
        tf.keras.layers.Rescaling(1./255, input_shape=(img_size, img_size, 3)),

        tf.keras.layers.Conv2D(32, 3, activation='relu'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(64, 3, activation='relu'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(128, 3, activation='relu'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(2, activation='softmax')  # 누적 모델과 동일
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss='sparse_categorical_crossentropy',  # 누적 모델과 동일
        metrics=['accuracy']
    )
    return model

def load_data(path, batch_size=32):
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = tf.keras.preprocessing.image_dataset_from_directory(
        path + "/train",
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=batch_size,
        label_mode='int'  # 누적 모델과 동일
    ).prefetch(AUTOTUNE)

    val_ds = tf.keras.preprocessing.image_dataset_from_directory(
        path + "/val",
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=batch_size,
        label_mode='int'  # 누적 모델과 동일
    ).prefetch(AUTOTUNE)

    return train_ds, val_ds

def train_and_save(phase_name, phase_path):
    print(f"\n{'='*50}")
    print(f"  {phase_name} 학습 시작")
    print(f"{'='*50}")

    train_ds, val_ds = load_data(phase_path)
    model = build_cnn()
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

    test_loss, test_acc = model.evaluate(test_ds)
    model.save(f"{SAVE_DIR}/model_{phase_name}.h5")

    results = {
        "dataset": phase_name,
        "train_acc":  history.history['accuracy'],
        "val_acc":    history.history['val_accuracy'],
        "train_loss": history.history['loss'],
        "val_loss":   history.history['val_loss'],
        "test_acc":   float(test_acc),
        "test_loss":  float(test_loss)
    }
    with open(f"{SAVE_DIR}/results_{phase_name}.json", "w") as f:
        json.dump(results, f, indent=2)

    print(f"  저장 완료 → val_acc: {max(history.history['val_accuracy']):.4f} / test_acc: {test_acc:.4f}")

In [ ]:
from pathlib import Path

def load_test_data(real_dir, fake_dir, image_size=(128, 128), batch_size=32):
    real_paths = list(Path(real_dir).glob("*.jpg")) + \
                 list(Path(real_dir).glob("*.png")) + \
                 list(Path(real_dir).glob("*.jpeg"))

    fake_paths = list(Path(fake_dir).glob("**/*.jpg")) + \
                 list(Path(fake_dir).glob("**/*.png")) + \
                 list(Path(fake_dir).glob("**/*.jpeg")) + \
                 list(Path(fake_dir).glob("**/*.webp"))

    print(f"Real: {len(real_paths)}장, Fake: {len(fake_paths)}장")

    all_paths = [str(p) for p in fake_paths + real_paths]
    all_labels = [0] * len(fake_paths) + [1] * len(real_paths)  # ai=0, real=1

    def preprocess(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, image_size)
        img = tf.cast(img, tf.float32)  # /255 없음 (모델 안에서 Rescaling)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((all_paths, all_labels))
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

test_ds = load_test_data(
    real_dir="/content/test_real/images",
    fake_dir="/content/test_fake/resized_data_Synthbuster/Synthbuster_Dataset",
)

Real: 9000장, Fake: 9000장


In [ ]:
train_and_save("phase1", "/content/clean/Phase1_CIFAKE")


  phase1 학습 시작
Found 89637 files belonging to 2 classes.
Found 23618 files belonging to 2 classes.
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


2802/2802 ━━━━━━━━━━━━━━━━━━━━ 72s 24ms/step - accuracy: 0.8246 - loss: 0.3858 - val_accuracy: 0.8748 - val_loss: 0.2952
Epoch 2/20
2802/2802 ━━━━━━━━━━━━━━━━━━━━ 60s 21ms/step - accuracy: 0.8917 - loss: 0.2658 - val_accuracy: 0.9133 - val_loss: 0.2171
Epoch 3/20
2802/2802 ━━━━━━━━━━━━━━━━━━━━ 63s 22ms/step - accuracy: 0.9133 - loss: 0.2199 - val_accuracy: 0.9249 - val_loss: 0.1911
Epoch 4/20
2802/2802 ━━━━━━━━━━━━━━━━━━━━ 60s 21ms/step - accuracy: 0.9240 - loss: 0.1932 - val_accuracy: 0.9259 - val_loss: 0.1877
Epoch 5/20
2802/2802 ━━━━━━━━━━━━━━━━━━━━ 58s 21ms/step - accuracy: 0.9331 - loss: 0.1732 - val_accuracy: 0.9325 - val_loss: 0.1711
Epoch 6/20
2802/2802 ━━━━━━━━━━━━━━━━━━━━ 58s 21ms/step - accuracy: 0.9402 - loss: 0.1566 - val_accuracy: 0.9422 - val_loss: 0.1506
Epoch 7/20
2802/2802 ━━━━━━━━━━━━━━━━━━━━ 59s 21ms/step - accuracy: 0.9458 - loss: 0.1418 - val_accuracy: 0.9442 - val_loss: 0.1445
Epoch 8/20
2802/2802 ━━━━━━━━━━━━━━━━━━━━ 62s 22ms/step - accuracy: 0.9496 - loss: 0.13

  저장 완료 → val_acc: 0.9544 / test_acc: 0.1772


In [ ]:
train_and_save("phase3", "/content/clean/Phase3_AI_REAL")


  phase3 학습 시작
Found 646 files belonging to 2 classes.
Found 180 files belonging to 2 classes.
Epoch 1/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 18s 627ms/step - accuracy: 0.8870 - loss: 0.3745 - val_accuracy: 0.8278 - val_loss: 0.5359
Epoch 2/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 9s 448ms/step - accuracy: 0.9226 - loss: 0.3099 - val_accuracy: 0.8278 - val_loss: 0.5660
Epoch 3/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 426ms/step - accuracy: 0.9226 - loss: 0.2985 - val_accuracy: 0.8278 - val_loss: 0.5208
Epoch 4/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 13s 563ms/step - accuracy: 0.9226 - loss: 0.2776 - val_accuracy: 0.8278 - val_loss: 0.5306
Epoch 5/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 11s 508ms/step - accuracy: 0.9226 - loss: 0.2394 - val_accuracy: 0.8278 - val_loss: 0.3870
Epoch 6/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 9s 450ms/step - accuracy: 0.9257 - loss: 0.2230 - val_accuracy: 0.8278 - val_loss: 0.4059
Epoch 7/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 467ms/step - accuracy: 0.9257 - loss: 0.1896 - val_accuracy: 0.8333 - val_loss: 0.4547
Epo

  저장 완료 → val_acc: 0.9056 / test_acc: 0.4744


In [ ]:
train_and_save("phase4", "/content/clean/Phase4_AI_REAL")


  phase4 학습 시작
Found 63960 files belonging to 2 classes.
Found 15990 files belonging to 2 classes.
Epoch 1/20
1999/1999 ━━━━━━━━━━━━━━━━━━━━ 194s 95ms/step - accuracy: 0.8737 - loss: 0.3079 - val_accuracy: 0.9183 - val_loss: 0.2191
Epoch 2/20
1999/1999 ━━━━━━━━━━━━━━━━━━━━ 178s 89ms/step - accuracy: 0.9167 - loss: 0.2194 - val_accuracy: 0.9287 - val_loss: 0.1911
Epoch 3/20
1999/1999 ━━━━━━━━━━━━━━━━━━━━ 161s 81ms/step - accuracy: 0.9275 - loss: 0.1936 - val_accuracy: 0.9338 - val_loss: 0.1741
Epoch 4/20
1999/1999 ━━━━━━━━━━━━━━━━━━━━ 182s 71ms/step - accuracy: 0.9346 - loss: 0.1766 - val_accuracy: 0.9405 - val_loss: 0.1588
Epoch 5/20
1999/1999 ━━━━━━━━━━━━━━━━━━━━ 142s 71ms/step - accuracy: 0.9410 - loss: 0.1590 - val_accuracy: 0.9423 - val_loss: 0.1524
Epoch 6/20
1999/1999 ━━━━━━━━━━━━━━━━━━━━ 142s 71ms/step - accuracy: 0.9452 - loss: 0.1478 - val_accuracy: 0.9455 - val_loss: 0.1475
Epoch 7/20
1999/1999 ━━━━━━━━━━━━━━━━━━━━ 157s 78ms/step - accuracy: 0.9493 - loss: 0.1368 - val_accur

In [ ]:
train_and_save("phase5", "/content/clean/Phase5_AI_REAL")
# model_comparsion에서 계속

장기 의존성 문제

In [ ]:
from pathlib import Path

def load_single_phase_val(phase_path, image_size=(128, 128), batch_size=32):
    ai_paths = list(Path(f"{phase_path}/val/ai").glob("*.jpg")) + \
               list(Path(f"{phase_path}/val/ai").glob("*.png"))
    real_paths = list(Path(f"{phase_path}/val/real").glob("*.jpg")) + \
                 list(Path(f"{phase_path}/val/real").glob("*.png"))

    all_paths = [str(p) for p in ai_paths + real_paths]
    all_labels = [0] * len(ai_paths) + [1] * len(real_paths)

    def preprocess(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, image_size)
        img = tf.cast(img, tf.float32)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((all_paths, all_labels))
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

for phase_name, phase_path in [
    ("Phase1", "/content/clean/Phase1_CIFAKE"),
    ("Phase3", "/content/clean/Phase3_AI_REAL"),
    ("Phase4", "/content/clean/Phase4_AI_REAL"),
    ("Phase5", "/content/clean/Phase5_AI_REAL"),
]:
    ds = load_single_phase_val(phase_path)
    loss, acc = sequential_model.evaluate(ds, verbose=0)
    print(f"{phase_name} val_acc: {acc:.4f}")

Phase1 val_acc: 0.4613
Phase3 val_acc: 0.7389
Phase4 val_acc: 0.4739
Phase5 val_acc: 0.9219
